# Laboratorio 1: Regresión Lineal Multivariable

**Estudiante:** [Tu Nombre]
**Dataset:** ATP Matches till 2022

En este laboratorio se implementa un modelo de regresión lineal multivariable para predecir la duración de los partidos de tenis basándose en diversas estadísticas.

In [ ]:
import os
import numpy as np
import pandas as pd
from matplotlib import pyplot
%matplotlib inline

## 1. Carga y Preprocesamiento de Datos

Utilizamos Pandas para cargar el dataset y filtrar las columnas necesarias para cumplir con $n \ge 10$ y $m \ge 10000$.

In [ ]:
df = pd.read_csv('../Datasets/1_ATP_matches_D/atp_matches_till_2022.csv')

# Seleccionamos columnas numéricas relevantes para la predicción de minutos
columns = [
    'draw_size', 'winner_ht', 'winner_age', 'loser_ht', 'loser_age', 
    'best_of', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 
    'w_2ndWon', 'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'minutes'
]

df = df[columns].dropna()

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")

# Separamos características (X) y variable objetivo (y)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
m = y.size

### 1.1 Normalización de Características

Es fundamental para que el descenso por gradiente converja eficientemente.

In [ ]:
def featureNormalize(X):
    X_norm = X.copy()
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = featureNormalize(X)

# Añadimos la columna de unos para el término de intercepción (theta_0)
X_norm = np.concatenate([np.ones((m, 1)), X_norm], axis=1)

## 2. Regresión Lineal

### 2.1 Función de Costo

In [ ]:
def computeCostMulti(X, y, theta):
    m = y.shape[0]
    h = np.dot(X, theta)
    J = (1 / (2 * m)) * np.sum(np.square(h - y))
    return J

### 2.2 Descenso por Gradiente

In [ ]:
def gradientDescentMulti(X, y, theta, alpha, num_iters):
    m = y.shape[0]
    J_history = []
    
    for i in range(num_iters):
        h = np.dot(X, theta)
        theta = theta - (alpha / m) * np.dot(X.T, (h - y))
        J_history.append(computeCostMulti(X, y, theta))
        
    return theta, J_history

## 3. Entrenamiento

In [ ]:
alpha = 0.01
num_iters = 1000

theta = np.zeros(X_norm.shape[1])
theta, J_history = gradientDescentMulti(X_norm, y, theta, alpha, num_iters)

# Graficar la convergencia del costo
pyplot.plot(np.arange(len(J_history)), J_history, lw=2)
pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J')
pyplot.title('Convergencia del Descenso por Gradiente')
print(f"Costo final: {J_history[-1]}")

## 4. Inferencia

Probamos el modelo con un ejemplo (promedio de los datos).

In [ ]:
# Ejemplo de inferencia con valores promedio
sample_data = np.mean(X, axis=0)
sample_norm = (sample_data - mu) / sigma
sample_norm = np.append(1, sample_norm) # Agregar bias

prediction = np.dot(sample_norm, theta)
print(f"Predicción de duración para valores promedio: {prediction:.2f} minutos")
print(f"Duración real promedio: {np.mean(y):.2f} minutos")